In [47]:
import torch
import os

In [2]:
tensor0d = torch.tensor(1)
tensor1d = torch.tensor([1, 2, 3])
tensor2d = torch.tensor([[1, 2],
                        [3, 4]])
tensor3d = torch.tensor([[[1, 2], [3, 4]],
                         [[5, 6], [7, 8]]])
print(tensor2d.dtype)

torch.int64


In [3]:
floatvec = torch.tensor([1.0, 2.0, 3.0])
print(floatvec.dtype)

torch.float32


In [4]:
floatvec = floatvec.to(torch.float64)
print(floatvec.dtype)

torch.float64


In [5]:
tensor3d.shape

torch.Size([2, 2, 2])

In [6]:
print(tensor3d.reshape(4, 2), "\n", "-"*180)
print(tensor3d.view(1, 1, 8))

tensor([[1, 2],
        [3, 4],
        [5, 6],
        [7, 8]]) 
 ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
tensor([[[1, 2, 3, 4, 5, 6, 7, 8]]])


In [7]:
print(tensor3d.T, "\n", "-"*180)
print(tensor2d.T)

tensor([[[1, 5],
         [3, 7]],

        [[2, 6],
         [4, 8]]]) 
 ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
tensor([[1, 3],
        [2, 4]])


C:\Users\98922\AppData\Local\Temp\ipykernel_42124\159892877.py:1: UserWarning: The use of `x.T` on tensors of dimension other than 2 to reverse their shape is deprecated and it will throw an error in a future release. Consider `x.mT` to transpose batches of matrices or `x.permute(*torch.arange(x.ndim - 1, -1, -1))` to reverse the dimensions of a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\TensorShape.cpp:4424.)
  print(tensor3d.T, "\n", "-"*180)


In [8]:
print(torch.matmul(tensor3d, tensor3d.T))
print(torch.matmul(tensor2d, tensor2d.T))

tensor([[[  7,  19],
         [ 15,  43]],

        [[ 34,  78],
         [ 46, 106]]])
tensor([[ 5, 11],
        [11, 25]])


In [9]:
print(tensor3d @ tensor3d.T)

tensor([[[  7,  19],
         [ 15,  43]],

        [[ 34,  78],
         [ 46, 106]]])


In [10]:
import torch.nn.functional as F
from torch.autograd import grad

y = torch.tensor([1.0])
x1 = torch.tensor([1.1])
w1 = torch.tensor([2.2], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)
z = x1 * w1 + b
a = torch.sigmoid(z)
loss = F.binary_cross_entropy(a, y)

grad_L_w1 = grad(loss, w1, retain_graph=True)
grad_L_b = grad(loss, b, retain_graph=True)

print(grad_L_w1)
print(grad_L_b)

(tensor([-0.0898]),)
(tensor([-0.0817]),)


In [11]:
loss.backward()
print(w1.grad)
print(b.grad)

tensor([-0.0898])
tensor([-0.0817])


In [12]:
class NeuralNetwork(torch.nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super().__init__()

        self.layers = torch.nn.Sequential(
            torch.nn.Linear(num_inputs, 30),
            torch.nn.ReLU(),

            torch.nn.Linear(30, 20),
            torch.nn.ReLU(),

            torch.nn.Linear(20, num_outputs)
        )

    def forward(self, x):
        logits = self.layers(x)
        return logits

In [13]:
model = NeuralNetwork(50, 3)
print(model)

NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=50, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=3, bias=True)
  )
)


In [14]:
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Number of model parameters: {num_params}")

Number of model parameters: 2213


In [15]:
print(model.layers[0].weight.shape)

torch.Size([30, 50])


In [16]:
torch.manual_seed(123)
X = torch.rand((1, 50))
out = model(X)
print(out)

tensor([[-0.1471,  0.0119, -0.3184]], grad_fn=<AddmmBackward0>)


In [17]:
with torch.no_grad():
    out = model(X)

print(out)

tensor([[-0.1471,  0.0119, -0.3184]])


In [18]:
with torch.no_grad():
    out = torch.softmax(model(X), dim=1)

print(out)

tensor([[0.3317, 0.3889, 0.2795]])


In [19]:
X_train = torch.tensor([
    [-1.2, 3.1],
    [-0.9, 2.9],
    [-0.5, 2.6],
    [2.3, -1.1],
    [2.7, -1.5]
])
y_train = torch.tensor([0, 0, 0, 1, 1])
X_test = torch.tensor([
    [-0.8, 2.8],
    [2.6, -1.6]
])
y_test = torch.tensor([0, 1])

In [20]:
from torch.utils.data import Dataset

class ToyDataset(Dataset):
    def __init__(self, X, y):
        self.features = X
        self.labels = y

    def __len__(self):
        return self.labels.shape[0]

    def __getitem__(self, idx):
        one_x = self.features[idx]
        one_y = self.labels[idx]
        return one_x, one_y

In [21]:
train_ds = ToyDataset(X_train, y_train)
test_ds = ToyDataset(X_test, y_test)

In [22]:
print(len(train_ds))

5


In [23]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    dataset=train_ds,
    num_workers=0,
    batch_size=2,
    shuffle=True
)
test_loader = DataLoader(
    dataset=test_ds,
    num_workers=0,
    batch_size=2,
    shuffle=False
)

In [24]:
for idx, (x, y) in enumerate(train_loader):
    print(f"Batch {idx+1}:", x, y)

Batch 1: tensor([[ 2.3000, -1.1000],
        [-0.5000,  2.6000]]) tensor([1, 0])
Batch 2: tensor([[-1.2000,  3.1000],
        [-0.9000,  2.9000]]) tensor([0, 0])
Batch 3: tensor([[ 2.7000, -1.5000]]) tensor([1])


In [25]:
train_loader = DataLoader(
    dataset=train_ds,
    batch_size=2,
    num_workers=0,
    shuffle=True,
    drop_last=True
)

In [26]:
for idx, (x, y) in enumerate(train_loader):
    print(f"Batch {idx+1}: ", x, y)

Batch 1:  tensor([[-0.5000,  2.6000],
        [ 2.3000, -1.1000]]) tensor([0, 1])
Batch 2:  tensor([[-1.2000,  3.1000],
        [ 2.7000, -1.5000]]) tensor([0, 1])


In [27]:
import torch.nn.functional as F
torch.manual_seed(28)
model = NeuralNetwork(num_inputs=2, num_outputs=2)
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)
num_epochs = 3
for epoch in range(num_epochs):
    model.train()
    for batch_idx, (features, labels) in enumerate(train_loader):
        logits = model(features)
        loss = F.cross_entropy(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        ##LOGGING
        print(f"Epoch: {epoch+1}/{num_epochs:03d}"
              f" | Batch {batch_idx}/{len(train_loader):03d}"
              f" | Train Loss: {loss:.2f}")

Epoch: 1/003 | Batch 0/002 | Train Loss: 0.75
Epoch: 1/003 | Batch 1/002 | Train Loss: 0.39
Epoch: 2/003 | Batch 0/002 | Train Loss: 0.31
Epoch: 2/003 | Batch 1/002 | Train Loss: 0.07
Epoch: 3/003 | Batch 0/002 | Train Loss: 0.04
Epoch: 3/003 | Batch 1/002 | Train Loss: 0.01


In [28]:
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Number of model parameters: {num_params}")

Number of model parameters: 752


In [29]:
model.eval()
with torch.no_grad():
    outputs = model(X_train)
print(outputs)

tensor([[ 1.9732, -2.4759],
        [ 1.8450, -2.3465],
        [ 1.6469, -2.1403],
        [-2.3899,  1.6433],
        [-2.8839,  2.0321]])


In [30]:
torch.set_printoptions(sci_mode=False)
probs = torch.softmax(outputs, dim=1)
print(probs)

tensor([[0.9884, 0.0116],
        [0.9851, 0.0149],
        [0.9778, 0.0222],
        [0.0174, 0.9826],
        [0.0073, 0.9927]])


In [31]:
preds = torch.argmax(probs, dim=1)
print(preds)

tensor([0, 0, 0, 1, 1])


In [32]:
preds = torch.argmax(outputs, dim=1)
print(preds)

tensor([0, 0, 0, 1, 1])


In [33]:
preds == y_train

tensor([True, True, True, True, True])

In [34]:
torch.sum(preds == y_train).item()

5

In [60]:
def compute_accuracy(model, dataloader, device):
    correct = 0.0
    total_examples = 0
    model = model.eval()
    with torch.no_grad():
        for features, labels in dataloader:
            features = features.to(device)
            labels = labels.to(device)
            logits = model(features)
            preds = torch.argmax(logits, dim=1)
            compare = preds == labels
            correct += torch.sum(compare)
            total_examples += len(compare)
    return (correct / total_examples).item()

In [36]:
print(f"train accuracy: {compute_accuracy(model, train_loader)}")

train accuracy: 1.0


In [37]:
print(f"test accuracy: {compute_accuracy(model, test_loader)}")

test accuracy: 1.0


In [38]:
torch.save(model.state_dict(), "pytorch_appendix.pth")

In [39]:
model = NeuralNetwork(2, 2)
model.load_state_dict(torch.load("pytorch_appendix.pth"))

<All keys matched successfully>

In [40]:
print(torch.cuda.is_available())

True


In [41]:
tens1 = torch.tensor([1.0, 2.0, 3.0])
tens2 = torch.tensor([1.0, 2.0, 3.0])
print(tens1 + tens2)

tensor([2., 4., 6.])


In [42]:
tens1 = torch.tensor([1.0, 2.0, 3.0]).to("cuda")
tens2 = torch.tensor([1.0, 2.0, 3.0]).to("cuda")
print(tens1 + tens2)

tensor([2., 4., 6.], device='cuda:0')


In [43]:
tens1 = tens1.to("cpu")
print(tens1 + tens2)

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!

In [44]:
torch.manual_seed(28)
model = NeuralNetwork(num_inputs=2, num_outputs=2)
device = torch.device("cuda")
model = model.to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)
num_epochs = 3
for epoch in range(num_epochs):
    model.train()
    for batch_idx, (features, labels) in enumerate(train_loader):
        features, labels = features.to(device), labels.to(device)
        logits = model(features)
        loss = F.cross_entropy(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    ##LOGGING
        print(f"Epoch: {epoch+1}/{num_epochs:03d}"
              f" | Batch {batch_idx}/{len(train_loader):03d}"
              f" | Train Loss: {loss:.2f}")

Epoch: 1/003 | Batch 0/002 | Train Loss: 0.75
Epoch: 1/003 | Batch 1/002 | Train Loss: 0.39
Epoch: 2/003 | Batch 0/002 | Train Loss: 0.31
Epoch: 2/003 | Batch 1/002 | Train Loss: 0.07
Epoch: 3/003 | Batch 0/002 | Train Loss: 0.04
Epoch: 3/003 | Batch 1/002 | Train Loss: 0.01


In [45]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [46]:
import torch.multiprocessing as mp
from torch.utils.data.distributed import DistributedSampler
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.distributed import init_process_group, destroy_process_group

In [48]:
def ddp_setup(rank, world_size):
    os.environ["MASTER_ADDR"] = "localhost"
    os.environ["MASTER_PORT"] = "12345"
    init_process_group(
        backend="nccl",
        rank=rank,
        world_size=world_size
    )
    torch.cuda.set_device(rank)

In [51]:
def prepare_dataset():
    train_loader = DataLoader(
        dataset=train_ds,
        batch_size=2,
        shuffle=False,
        pin_memory=True,
        drop_last=True,
        sampler=DistributedSampler(train_ds)
    )
    test_loader = DataLoader(
        dataset=test_ds,
        batch_size=2,
        shuffle=False,
        drop_last=False,
        pin_memory=True,
        sampler=DistributedSampler(test_ds, shuffle=False)
    )

    return train_loader, test_loader

In [54]:
def main(rank, world_size, num_epochs):
    ddp_setup(rank, world_size)
    train_loader, test_loader = prepare_dataset()
    model = NeuralNetwork(num_inputs=2, num_outputs=2).to(rank)
    model = DDP(model, device_ids=[rank])
    optimizer = torch.optim.SGD(model.parameters(), lr=0.5)
    for epoch in range(num_epochs):
        model.train()
        for features, labels in train_loader:
            features, labels = features.to(rank), labels.to(rank)
            ##LOGGING
            print(f"[GPU{rank}] Epoch: {epoch+1:03d}/{num_epochs:03d}"
              f" | Batchsize {labels.shape[0]:03d}"
              f" | Train/Val Loss: {loss:.2f}")

    model.eval()
    train_acc = compute_accuracy(model, train_loader, device=rank)
    print(f"[GPU{rank}] Training Accuracy", train_acc)
    test_acc = compute_accuracy(model, test_loader, device=rank)
    print(f"[GPU{rank}] Test Accuracy", test_acc)
    destroy_process_group()

In [59]:
if __name__ == "main":
    print(f"Number GPUs available on Device: {torch.cuda.device_count()}")
    torch.manual_seed(28)
    num_epochs = 3
    world_size = torch.cuda.device_count()
    mp.spawn(main, args=(world_size, num_epochs), nprocs=world_size)

In [ ]:
b = torch.randn(10000, 10000).to("cuda")

In [38]:
b = torch.randn(10000, 10000).to("cuda")

In [ ]:
%timeit a @ b